# 02 Lidar DEM derivatives with Dask Gateway

This notebook computes fault-sensitive terrain derivatives from USGS 3DEP 1 m lidar DEMs and aggregates them to the 100 m analysis grid from notebook 01.

**Why aggregate.** Fault scarps are often only a few metres high. A 100 m DEM does not show them. The notebook computes derivatives at 1 m or 2 m and keeps sub-cell structure with statistics such as the maximum slope anomaly and the structure tensor.

**Why Dask Gateway.** Each 3DEP 1 m tile is about 10 km by 10 km, or about 400 MB. The tiles are independent, so the work spreads across Gateway workers. Each worker reads its tile directly from the USGS cloud storage and returns a small 100 m result. Only the notebook server writes to the store.

**Inputs.** `grid.json` from notebook 01. 3DEP 1 m tiles found through the USGS TNM Access API.

**Output.** `dem1m_features.zarr` under `STORE_ROOT`.

| Layer | Meaning |
|---|---|
| `elev` | Mean elevation |
| `slope_mean`, `slope_max` | Slope in degrees |
| `slope_anom_max` | Maximum of slope minus local mean slope (scarp indicator) |
| `lap_min`, `lap_max` | Laplacian extremes (paired convex and concave breaks at scarps) |
| `tpi{S}_rms` | RMS topographic position index at scale S metres |
| `lin_coherence`, `lin_sin2`, `lin_cos2` | Structure tensor coherence and strike of linear features (axial, doubled angle) |
| `elev_detrended`, `elev_detrended_slope` | Elevation minus a 5 km smooth surface, and its slope |
| `cov` | Fraction of the cell covered by lidar |

**Coverage.** 1 m lidar does not cover all of Nevada. Cells without lidar are NaN. LightGBM in notebook 03 handles NaN values. `cov` tells the model where the lidar exists.

**Annotations.** `VERIFY:` marks an untested assumption about an external service. `NOTE:` marks a design decision.

**Behavior on errors.** If Dask Gateway is not available, the notebook stops.

#### What the next cell does

It loads the libraries that the notebook uses. Nothing is downloaded or computed yet.

- **`fsspec`** reads `grid.json` from the store, which can be a local folder or S3.
- **`rasterio`** opens the lidar tiles directly over the internet, without downloading whole files first. Its helpers:
  - `Resampling` names the ways to combine or interpolate cells.
  - `array_bounds` and `from_origin` convert between rows and columns and map coordinates.
  - `calculate_default_transform` and `reproject` change a raster's coordinate system or cell size.
  - `transform_bounds` converts a box between coordinate systems, and `warp_transform` converts individual points.
  - `WarpedVRT` reprojects a raster on the fly while it's read.
- **`affine`** rescales the position rule of a raster when it's read at a coarser resolution.
- **`rioxarray` and `xarray`** hold the output grid. `rioxarray` adds the `.rio` methods.
- **`numpy`, `pandas`, `scipy.ndimage`, `requests`, and `matplotlib`** handle arrays, the tile table, image filters, web requests, and plots.

In [ ]:
import json
import os
import pathlib
import re
import time
import warnings

import fsspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import requests
import rioxarray  # noqa: F401
import xarray as xr
from affine import Affine
from rasterio.enums import Resampling
from rasterio.transform import array_bounds, from_origin
from rasterio.warp import calculate_default_transform, reproject, transform_bounds
from rasterio.warp import transform as warp_transform
from rasterio.vrt import WarpedVRT
from scipy import ndimage

## 0. Configuration

#### What the next cell does

It only sets values. Edit this cell to change how the notebook runs.

- **`STORE_ROOT` and `OUTPUT_NAME`**: where the notebook reads `grid.json` and writes `dem1m_features.zarr`. They must be the same store that notebook 01 used.
- **TNM search.** TNM (The National Map) is the USGS service that lists the 3DEP elevation products. `DEM_DATASET` names the 1 m lidar DEM, and `TNM_PAGE` is the number of records per request.
- **`PARAMS`**, the terrain calculation settings:
  - `decimate`: reads the 1 m tile at a coarser resolution. With `2`, each value is the average of a 2 × 2 block, which gives 2 m and uses a quarter of the memory. `1` keeps full resolution.
  - `tpi_scales_m`: the neighborhood sizes (25 m and 100 m) for the topographic position index (TPI). TPI measures how much higher or lower a point is than its surroundings.
  - `smooth_sigma_m`: 3 m of smoothing before slopes and curvature are computed. This suppresses lidar noise, such as vegetation and small pits.
  - `slope_anom_window_m`: the 200 m neighborhood that each slope is compared with.
  - `geographic_work_res_m`: the working resolution, used only for a tile stored in degrees instead of metres.
- **`MAX_TILES`**: processes only the first 8 tiles as a test. Set it to `None` for the whole area.
- **Cluster settings.**
  - `GATEWAY_OPTIONS`: options for the Dask Gateway cluster, such as the worker size. An empty `{}` uses the Gateway's defaults. Section 4 shows how to list the allowed values.
  - `MIN_WORKERS` and `MAX_WORKERS`: the range within which the cluster grows and shrinks.
- **`GDAL_ENV`**: settings for reading files over HTTP.
  - Don't list the remote folder, which saves a request per file.
  - Only allow `.tif` files.
  - Retry failed requests 5 times, 2 seconds apart.
  - Don't cache what has been read.

In [ ]:
# Default: a "gems" folder next to the notebooks ($PWD/gems; a Jupyter kernel runs in the notebook's folder).
STORE_ROOT = os.environ.get("GEMS_STORE", str(pathlib.Path.cwd() / "gems"))
OUTPUT_NAME = "dem1m_features.zarr"

TNM_API = "https://tnmaccess.nationalmap.gov/api/v1/products"
DEM_DATASET = "Digital Elevation Model (DEM) 1 meter"     # VERIFY: TNM dataset name
TNM_PAGE = 500

PARAMS = {
    "decimate": 2,             # NOTE: 1 = full 1 m (about 16 GB per worker); 2 = 2 m (about 4 GB)
    "tpi_scales_m": (25.0, 100.0),
    "smooth_sigma_m": 3.0,     # smoothing before gradients, Laplacian, and structure tensor
    "slope_anom_window_m": 200.0,
    "geographic_work_res_m": 10.0,   # only used if a tile is in a geographic CRS
}

MAX_TILES = 8              # NOTE: test with a few tiles first. Set None for the full region.
# Dask Gateway cluster options, passed to gateway.new_cluster(). {} = the Gateway defaults.
# To see the option names and allowed values, run gateway.cluster_options() in a cell (section 4).
# Example: {"worker_resource_allocation": "<one of the allowed values>"}
# NOTE: each worker needs about 4 GB with decimate = 2 and about 16 GB with decimate = 1.
GATEWAY_OPTIONS = {}
MIN_WORKERS, MAX_WORKERS = 2, 40

GDAL_ENV = {
    "GDAL_DISABLE_READDIR_ON_OPEN": "EMPTY_DIR",
    "CPL_VSIL_CURL_ALLOWED_EXTENSIONS": ".tif,.tiff",
    "GDAL_HTTP_MAX_RETRY": "5",
    "GDAL_HTTP_RETRY_DELAY": "2",
    "VSI_CACHE": "FALSE",
}

#### What the next cell does

It loads the grid that notebook 01 created, so the results of this notebook line up exactly with the other layers.

1. It opens the store and reads `grid.json`. From it, `H` and `W` are the number of rows and columns, and `RES` is the cell size (100 m).
2. `make_template` rebuilds the empty 100 m grid, with the same cell-centre coordinates as in notebook 01.
3. It converts the grid's corners from UTM to longitude and latitude, because the TNM search needs the area in longitude and latitude. It samples 51 points per edge, so that the curved edges are fully covered.
4. It prints the grid size and the area.

In [ ]:
fs, _ = fsspec.core.url_to_fs(STORE_ROOT)
def store_path(rel):
    return f"{STORE_ROOT.rstrip('/')}/{rel}"

with fs.open(store_path("grid.json")) as f:
    GRID = json.load(f)
H, W, RES = GRID["height"], GRID["width"], GRID["res"]

def make_template(grid):
    x = grid["xmin"] + grid["res"] * (np.arange(grid["width"]) + 0.5)
    y = grid["ymax"] - grid["res"] * (np.arange(grid["height"]) + 0.5)
    da = xr.DataArray(np.full((grid["height"], grid["width"]), np.nan, np.float32),
                      coords={"y": y, "x": x}, dims=("y", "x"))
    tr = from_origin(grid["xmin"], grid["ymax"], grid["res"], grid["res"])
    return da.rio.write_crs(grid["crs"]).rio.write_transform(tr)

template = make_template(GRID)
AOI_LONLAT = transform_bounds(GRID["crs"], "EPSG:4326", GRID["xmin"], GRID["ymax"] - H * RES,
                              GRID["xmin"] + W * RES, GRID["ymax"], densify_pts=51)
print(f"Grid {W} x {H}, AOI {tuple(round(v, 3) for v in AOI_LONLAT)}")

## 1. Find the 3DEP 1 m tiles

The TNM Access API returns one record per tile and lidar project. Where two projects cover the same tile, the notebook keeps the most recent publication.

#### What the next cell does

It finds the 1 m lidar tiles that cover the grid, and keeps one version of each tile.

1. **Search.** `tnm_products` asks the TNM API for all 1 m DEM GeoTIFFs that overlap the area. The API returns results in pages, so the function keeps asking for the next page, starting where the last one ended (`offset`), until it has every record.
2. **Why de-duplicate.** A 3DEP 1 m tile is a 10 km × 10 km square. When two lidar projects covered the same square, TNM lists both. The notebook needs one ID per square, a **tile key**, so it can keep only one version.
3. **Tile key from the file name.** Most file names contain the UTM zone and the tile index, for example `USGS_1M_11_x30y452_...`. The key is then `11_x30y452`.
4. **Older file names.** Some older projects use names like `USGS_one_meter_x28y435_<project>.tif`, with no zone. `infer_tile_key` works out the zone:
   - The `x` index is the tile's easting (its east–west UTM coordinate) in 10 km steps.
   - It converts the tile's centre into the nearby UTM zones and picks the zone in which the easting matches that index.
   - The zone can't simply be read from the longitude, because tiles extend past zone boundaries.
5. **No key.** Any record that still has no key gets its download address as its key, and a warning. Such tiles can't be de-duplicated.
6. **Keep the newest.** It sorts by publication date and keeps the most recent record for each tile key.
7. It prints the number of records, the number of unique tiles, and a rough data volume (about 0.4 GB per tile). With `MAX_TILES` set, it keeps only the first tiles.

In [ ]:
def tnm_products(bbox_lonlat, dataset, page=TNM_PAGE):
    items, offset = [], 0
    while True:
        r = requests.get(TNM_API, params={"datasets": dataset, "bbox": ",".join(f"{v:.5f}" for v in bbox_lonlat),
                                          "prodFormats": "GeoTIFF", "max": page, "offset": offset}, timeout=120)
        r.raise_for_status()
        j = r.json()                                   # VERIFY: response keys "items" and "total"
        batch = j.get("items", [])
        items += batch
        if not batch or len(items) >= int(j.get("total", len(items))):
            break
        offset += len(batch)
    return pd.DataFrame(items)

prod = tnm_products(AOI_LONLAT, DEM_DATASET)
if prod.empty:
    raise RuntimeError("TNM returned no 1 m DEM tiles for the AOI. Check DEM_DATASET and the AOI.")

# Tile key such as "11_x30y452" (UTM zone + tile index), taken from the download URL.
key = prod["downloadURL"].str.extract(r"USGS_1M_(\d+)_(x\d+y\d+)", expand=True)
prod["tile_key"] = key[0] + "_" + key[1]

def infer_tile_key(url, bbox):
    # Older files (USGS_one_meter_x28y435_<project>.tif) have no zone in the name. Tiles extend past
    # zone edges, so pick the zone in which the tile centre has the easting implied by x (10 km units).
    m = re.search(r"_x(\d+)y(\d+)_", url)
    if not m or not isinstance(bbox, dict):
        return None
    lon, lat = (bbox["minX"] + bbox["maxX"]) / 2, (bbox["minY"] + bbox["maxY"]) / 2
    target = int(m.group(1)) * 10_000 + 5_000
    z0 = int((lon + 180) // 6) + 1
    zone = min(range(z0 - 1, z0 + 2),
               key=lambda z: abs(warp_transform("EPSG:4326", f"EPSG:326{z:02d}", [lon], [lat])[0][0] - target))
    return f"{zone}_x{m.group(1)}y{m.group(2)}"

old_style = prod["tile_key"].isna()
prod.loc[old_style, "tile_key"] = [infer_tile_key(u, b) for u, b in
                                   zip(prod.loc[old_style, "downloadURL"], prod.loc[old_style, "boundingBox"])]
n_nokey = int(prod["tile_key"].isna().sum())
if n_nokey:
    warnings.warn(f"{n_nokey} records have no tile key. They are kept without de-duplication.")
    prod.loc[prod.tile_key.isna(), "tile_key"] = prod.loc[prod.tile_key.isna(), "downloadURL"]
prod["publicationDate"] = pd.to_datetime(prod.get("publicationDate"), errors="coerce")
tiles = prod.sort_values("publicationDate").groupby("tile_key").tail(1).reset_index(drop=True)
print(f"{len(prod)} records, {len(tiles)} unique tiles, about {len(tiles) * 0.4:.0f} GB to read")
if MAX_TILES:
    tiles = tiles.head(MAX_TILES)
    print(f"MAX_TILES = {MAX_TILES}: processing a test subset")

## 2. Tile functions

`tile_derivatives` is pure NumPy, so you can test it in the notebook. `aggregate_to_grid` warps each derivative into the part of the 100 m grid that the tile covers, with a resampling method that fits the statistic. `process_tile` reads one tile and calls both functions. It runs on the Gateway workers.

#### What the next cell does

It defines the three functions that process one lidar tile. It only defines them; they run in the next sections.

**`tile_derivatives(z, res, p)`** computes terrain measures at the tile's own resolution (2 m by default):

- **Gaps.** It skips tiles that are nearly empty. It temporarily fills gaps with the mean elevation, because the filters below can't handle missing values.
- **Smoothing.** A 3 m Gaussian smooth removes lidar noise before slopes are computed.
- **Slope.** `np.gradient` gives the elevation change per metre across columns and rows. Rows increase southward, so the north component is the negative of the row gradient. `slope` is the steepness in degrees.
- **Laplacian (curvature).** It is positive in hollows (concave ground) and negative on crests (convex ground). A fault scarp has a convex top edge and a concave foot, so it produces a paired minimum and maximum.
- **Slope anomaly.** This is the slope minus the average slope within 200 m. It picks out short, steep steps, such as scarps, against the general steepness of the hillside.
- **Masking near gaps.** Cells whose neighborhood reaches into a gap would be computed partly from the filled values, so they are set to NaN.
- **Structure tensor.** It stores the products of the east and north slope components: `st_ee`, `st_nn`, and `st_en`. Averaged over a 100 m cell, they show whether the slopes in that cell mostly face one direction, as they do along a scarp or a ridge. Section 6 turns them into a direction and a strength.
- **TPI.** This is the elevation minus the mean elevation within 25 m and within 100 m, squared. It measures how bumpy or stepped the surface is at each scale.
- **Coverage.** `cov_sum` counts the cells that have lidar data.

Each output layer also names the rule used to shrink it to 100 m: `average`, `max`, `min`, or `sum`.

**`aggregate_to_grid(layers, ...)`** shrinks the tile's layers onto the 100 m grid:

1. It finds which rows and columns of the grid the tile covers, and returns nothing if the tile lies outside the grid.
2. It resamples each layer into that window, using the layer's own rule. For example, `slope_anom_max` keeps the **strongest** scarp signal inside each 100 m cell. Averaging would blur a 10 m-wide scarp into its surroundings.
3. It turns the count of lidar cells into a coverage fraction from 0 to 1.
4. It returns the window's position and its small arrays: about 100 × 100 cells for a 10 km tile.

**`process_tile(url, ...)`** does the whole job for one tile. It runs on the workers.

1. It opens the tile over HTTPS with the `GDAL_ENV` settings. It stops if the tile has no coordinate system.
2. **Tile in degrees:** it reprojects the tile to UTM at 10 m while reading it (`WarpedVRT`).
3. **Normal tile:** it reads the tile at reduced resolution, averaging each `decimate` × `decimate` block. If the file has built-in lower-resolution copies (overviews), rasterio reads those instead, which downloads less. It then adjusts the position rule to the new cell size.
4. It runs `tile_derivatives` and `aggregate_to_grid`, and adds the tile's address and processing time. Only this small result goes back to the notebook.

In [ ]:
def _odd(n):
    n = max(int(round(n)), 3)
    return n if n % 2 else n + 1

def tile_derivatives(z, res, p):
    """Terrain derivatives at native resolution. Returns a list of (name, array, resampling)."""
    valid = np.isfinite(z)
    if valid.mean() < 0.01:
        return None
    zf = np.where(valid, z, np.nanmean(z)).astype(np.float32)
    s = max(p["smooth_sigma_m"] / res, 0.5)
    zs = ndimage.gaussian_filter(zf, s)
    gr, gc = np.gradient(zs, res)
    ge, gn = gc, -gr                                   # east and north components
    slope = np.degrees(np.arctan(np.hypot(ge, gn))).astype(np.float32)
    lap = (ndimage.laplace(zs) / (res * res)).astype(np.float32)
    win = _odd(p["slope_anom_window_m"] / res)
    slope_anom = slope - ndimage.uniform_filter(slope, win)

    # Cells near nodata get unreliable derivatives. Mask them.
    reach = _odd(max(max(p["tpi_scales_m"]), p["slope_anom_window_m"]) / res + 6 * s)
    near_gap = ndimage.uniform_filter(valid.astype(np.float32), reach) < 0.999
    def m(a):
        return np.where(near_gap, np.nan, a).astype(np.float32)

    out = [("elev", np.where(valid, z, np.nan).astype(np.float32), "average"),
           ("slope_mean", m(slope), "average"), ("slope_max", m(slope), "max"),
           ("slope_anom_max", m(slope_anom), "max"),
           ("lap_min", m(lap), "min"), ("lap_max", m(lap), "max"),
           ("st_ee", m(ge * ge), "average"), ("st_nn", m(gn * gn), "average"), ("st_en", m(ge * gn), "average")]
    for sc in p["tpi_scales_m"]:
        tpi = zf - ndimage.uniform_filter(zf, _odd(sc / res))
        out.append((f"tpi{int(sc)}_ms", m(tpi * tpi), "average"))
    out.append(("cov_sum", valid.astype(np.float32), "sum"))   # NOTE: needs GDAL >= 3.1 for "sum"
    return out

def aggregate_to_grid(layers, src_transform, src_crs, grid):
    h, w = layers[0][1].shape
    left, bottom, right, top = array_bounds(h, w, src_transform)
    bx0, by0, bx1, by1 = transform_bounds(src_crs, grid["crs"], left, bottom, right, top, densify_pts=21)
    res = grid["res"]
    c0 = max(int(np.floor((bx0 - grid["xmin"]) / res)), 0)
    c1 = min(int(np.ceil((bx1 - grid["xmin"]) / res)), grid["width"])
    r0 = max(int(np.floor((grid["ymax"] - by1) / res)), 0)
    r1 = min(int(np.ceil((grid["ymax"] - by0) / res)), grid["height"])
    if c1 <= c0 or r1 <= r0:
        return None
    dst_transform = from_origin(grid["xmin"] + c0 * res, grid["ymax"] - r0 * res, res, res)
    src_res = abs(src_transform.a)
    out = {}
    for name, arr, method in layers:
        dst = np.full((r1 - r0, c1 - c0), np.nan if method != "sum" else 0.0, np.float32)
        reproject(arr, dst, src_transform=src_transform, src_crs=src_crs,
                  dst_transform=dst_transform, dst_crs=grid["crs"],
                  resampling=getattr(Resampling, method),
                  src_nodata=None if method == "sum" else np.nan,
                  dst_nodata=None if method == "sum" else np.nan)
        out[name] = dst
    out["cov"] = np.clip(out.pop("cov_sum") / (res / src_res) ** 2, 0, 1).astype(np.float32)
    return {"row0": r0, "col0": c0, "layers": out}

def process_tile(url, grid, p, gdal_env):
    t_start = time.time()
    with rasterio.Env(**gdal_env):
        with rasterio.open(url) as src:
            if src.crs is None:
                raise ValueError(f"No CRS: {url}")
            if src.crs.is_geographic:
                tr, w, h = calculate_default_transform(src.crs, grid["crs"], src.width, src.height,
                                                       *src.bounds, resolution=p["geographic_work_res_m"])
                with WarpedVRT(src, crs=grid["crs"], transform=tr, width=w, height=h,
                               resampling=Resampling.bilinear) as vrt:
                    z = vrt.read(1, masked=True).astype(np.float32).filled(np.nan)
                    transform, crs = tr, vrt.crs
            else:
                f = int(p["decimate"])
                oh, ow = src.height // f, src.width // f
                # Decimated reads use COG overviews when they exist, which cuts the download.
                z = src.read(1, out_shape=(oh, ow), masked=True,
                             resampling=Resampling.average if f > 1 else Resampling.nearest)
                z = z.astype(np.float32).filled(np.nan)
                transform = src.transform * Affine.scale(src.width / ow, src.height / oh)
                crs = src.crs
    lyr = tile_derivatives(z, abs(transform.a), p)
    if lyr is None:
        return None
    res = aggregate_to_grid(lyr, transform, crs, grid)
    if res is not None:
        res["url"], res["seconds"] = url, time.time() - t_start
    return res

## 3. Test one tile on the notebook server

Run one tile before you start the cluster. This checks remote access to the 3DEP bucket and the memory use for your `decimate` value.

#### What the next cell does

It processes one tile on the notebook server, before any cluster is started. This test shows:

- **Access.** The server can read the 3DEP files from the USGS cloud storage.
- **Time and memory.** How long one tile takes, and whether your `decimate` setting fits in memory.

It runs `process_tile` on the first tile in the list, and stops if that tile returns no data. It prints the processing time and the size and position of the tile's window in the grid. Then it shows three images of the result: elevation, slope anomaly, and lidar coverage.

The elevation should look like terrain. The slope anomaly should show sharp linear features, such as scarps and channel banks. Coverage should be close to 1 wherever there's lidar.

In [ ]:
test = process_tile(tiles.downloadURL.iloc[0], GRID, PARAMS, GDAL_ENV)
if test is None:
    raise RuntimeError("The test tile returned no data. Try another tile.")
print(f"{test['seconds']:.0f} s, window {test['layers']['cov'].shape} at row {test['row0']}, col {test['col0']}")
fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
for ax, n in zip(axes, ["elev", "slope_anom_max", "cov"]):
    ax.imshow(test["layers"][n]); ax.set_title(n); ax.set_axis_off()

## 4. Start the Dask cluster

The next cell starts a Dask Gateway cluster and connects to it.

**Choosing the worker size.** To see which options the Gateway offers, and their allowed values, run this in a new cell:

```python
from dask_gateway import Gateway
Gateway().cluster_options()
```

Then put the values you want in `GATEWAY_OPTIONS` in section 0.

**Version check.** `client.get_versions(check=True)` stops if the workers run different versions of Python or of the Dask packages than this notebook. A mismatch causes errors that are hard to diagnose later. If it stops, run this notebook in the image's default kernel. If the versions still differ, the workers are using a different image from your server. Add `"image": os.environ["JUPYTER_IMAGE_SPEC"]` to `GATEWAY_OPTIONS` to start them in your server's image.

#### What the next cell does

It starts the cluster of workers that will process the tiles in parallel.

1. **Connect.** `Gateway()` connects to the GeoLab Dask Gateway.
2. **Create the cluster.** `gateway.new_cluster(**GATEWAY_OPTIONS)` creates a cluster with the options from section 0. It raises an error if an option name isn't offered or a value isn't allowed.
3. **Adaptive scaling.** `cluster.adapt` turns on adaptive scaling. The Gateway adds workers while tiles are waiting and removes idle ones, always between `MIN_WORKERS` and `MAX_WORKERS`.
4. **Client.** `client` is the notebook's connection to the cluster. The dashboard link opens a live view of the workers and tasks.
5. **Wait.** It waits up to 10 minutes for the first worker to start.
6. **Version check.** `client.get_versions(check=True)` compares the versions of Python and the core Dask packages on the notebook, the scheduler (the process that hands out tasks), and the workers. It stops if they differ, because mismatches cause errors that are hard to trace later. Section 4 explains what to do if this happens.

In [ ]:
from dask.distributed import as_completed
from dask_gateway import Gateway

gateway = Gateway()
cluster = gateway.new_cluster(**GATEWAY_OPTIONS)
cluster.adapt(minimum=MIN_WORKERS, maximum=MAX_WORKERS)
client = cluster.get_client()
print("Dashboard:", cluster.dashboard_link)

client.wait_for_workers(1, timeout=600)
client.get_versions(check=True)   # stops if the workers run different package versions

## 5. Process all tiles and build the mosaic

Tiles overlap slightly at their edges. Where two results cover the same 100 m cell, the mosaic keeps the value from the tile with the higher lidar coverage for that cell. Failed tiles are listed at the end. They do not stop the run.

#### What the next cell does

It sends every tile to the workers and assembles their results into full-grid maps (a mosaic).

1. **Empty mosaic.** For each layer name from the test tile, it creates a full-size grid filled with NaN. `best` records, for each cell, the highest lidar coverage seen so far.
2. **`add_to_mosaic(res)`** places one tile's result:
   - It finds the tile's window in the grid.
   - Neighboring tiles overlap slightly at their edges. Wherever this tile's coverage is **higher** than the best so far, its values replace the old ones. Each cell therefore takes all its values from the one tile that covers it best, not a mix of tiles.
3. **Submit.** `client.map` creates one task per tile address and sends them to the workers.
   - `retries=2` retries a failed tile twice, for example after a network error.
   - `pure=False` always runs the task, even if an identical task ran before.
4. **Collect.** `as_completed` hands back each result as soon as its tile finishes, in any order. For each tile:
   - **Error:** it's recorded in `failed`, and the loop continues. One bad tile doesn't stop the run.
   - **Success:** the result is added to the mosaic.
   - Either way, `fut.release()` frees the task's memory on the cluster.
5. It prints progress every 25 tiles, and at the end the number of finished and failed tiles, with the first 20 errors.

In [ ]:
names = list(test["layers"].keys())
mosaic = {n: np.full((H, W), np.nan, np.float32) for n in names}
best = np.zeros((H, W), np.float32)

def add_to_mosaic(res):
    r0, c0 = res["row0"], res["col0"]
    L = res["layers"]
    h, w = L["cov"].shape
    sl = (slice(r0, r0 + h), slice(c0, c0 + w))
    cov = np.nan_to_num(L["cov"])
    take = cov > best[sl]
    best[sl][take] = cov[take]
    for n in names:
        mosaic[n][sl][take] = L[n][take]

futures = client.map(process_tile, list(tiles.downloadURL), grid=GRID, p=PARAMS,
                     gdal_env=GDAL_ENV, retries=2, pure=False)
failed, done = [], 0
for fut in as_completed(futures):
    try:
        res = fut.result()
    except Exception as e:
        failed.append((fut.key, repr(e)))
        continue
    finally:
        fut.release()
    if res is not None:
        add_to_mosaic(res)
    done += 1
    if done % 25 == 0:
        print(f"{done}/{len(futures)} tiles")
print(f"Finished: {done} tiles, {len(failed)} failed")
for k, e in failed[:20]:
    print("  FAILED", k, e)

## 6. Derived layers and output

#### What the next cell does

It defines two helper functions for section 6. It only defines them.

**`nan_gaussian(a, sigma)`** smooths a map that has gaps, without letting the gaps pull the values down:

1. It smooths the values, with the gaps set to 0.
2. It separately smooths a map that is 1 where there's data and 0 in the gaps.
3. It divides the first result by the second. This gives the weighted average of only the valid neighbors.
4. Where almost no valid neighbors exist, the result is NaN.

**`structure_tensor_features(see, snn, sen)`** turns the averaged slope products from each 100 m cell into two things:

- **Coherence**, from 0 to 1. It is 1 when all the slopes in the cell face the same direction, as along a scarp, a ridge, or a straight valley wall. It is near 0 when the slopes face every direction, as on rough or flat ground.
- **Orientation of the linear feature.** The dominant slope direction points across the feature, so the feature itself runs 90° from it. The orientation is stored as the sine and cosine of **twice** the angle. A line striking 10° is the same line as one striking 190°, and 179° is next to 1°. Doubling the angle makes such directions equal or close, and sine and cosine avoid the jump at 180°. The sign change converts the slope direction into the feature direction.

In [ ]:
def nan_gaussian(a, sigma):
    m = np.isfinite(a)
    num = ndimage.gaussian_filter(np.where(m, a, 0).astype(np.float32), sigma)
    den = ndimage.gaussian_filter(m.astype(np.float32), sigma)
    with np.errstate(invalid="ignore", divide="ignore"):
        out = num / den
    out[den < 1e-3] = np.nan
    return out

def structure_tensor_features(see, snn, sen, eps=1e-12):
    """Coherence (0..1) and doubled-angle strike of linear features.

    The dominant gradient direction is theta = 0.5 * atan2(2 sen, see - snn), from east.
    Linear features strike at theta + 90 degrees, so the doubled angle changes sign.
    """
    coh = np.sqrt((see - snn) ** 2 + 4 * sen ** 2) / (see + snn + eps)
    two_theta = np.arctan2(2 * sen, see - snn)
    return coh.astype(np.float32), (-np.sin(two_theta)).astype(np.float32), (-np.cos(two_theta)).astype(np.float32)

#### What the next cell does

It computes the final lidar layers and writes them to `dem1m_features.zarr`.

1. **Direct layers.** It copies these from the mosaic unchanged: elevation, mean and maximum slope, maximum slope anomaly, the Laplacian minimum and maximum, and coverage.
2. **TPI.** The workers averaged squared TPI values. The square root turns these into a root-mean-square value, a typical size of the bumps at each scale.
3. **Linear features.** `structure_tensor_features` gives `lin_coherence`, `lin_sin2`, and `lin_cos2`.
4. **Detrended elevation.** This is the elevation minus a version smoothed over 5 km. It removes the broad shape of valleys and ranges and keeps sharper steps, such as range-front faults. `elev_detrended_slope` is the steepness of that detrended surface.
5. **Dataset.** It puts every layer into an xarray `Dataset` named `dem_<layer>`, on the grid's coordinates. It records the coordinate system, the grid, the settings, and the counts of processed and failed tiles as attributes.
6. **Write.** It writes the dataset to zarr in 1024 × 1024 chunks. It writes the arrays directly from the notebook's memory. Dask isn't used for the write, because it would send the data to the Gateway workers, and they can't reach a local `STORE_ROOT`.
7. It prints the share of the grid where more than half of each cell is covered by lidar.

In [ ]:
out = {}
for n in ["elev", "slope_mean", "slope_max", "slope_anom_max", "lap_min", "lap_max", "cov"]:
    out[n] = mosaic[n]
for sc in PARAMS["tpi_scales_m"]:
    out[f"tpi{int(sc)}_rms"] = np.sqrt(mosaic[f"tpi{int(sc)}_ms"])
out["lin_coherence"], out["lin_sin2"], out["lin_cos2"] = structure_tensor_features(
    mosaic["st_ee"], mosaic["st_nn"], mosaic["st_en"])

# NOTE: 5 km detrending removes basin-scale relief and keeps range-front steps.
out["elev_detrended"] = mosaic["elev"] - nan_gaussian(mosaic["elev"], 5000.0 / RES)
gr, gc = np.gradient(out["elev_detrended"], RES)
out["elev_detrended_slope"] = np.degrees(np.arctan(np.hypot(gr, gc))).astype(np.float32)

dem = xr.Dataset({f"dem_{k}": (("y", "x"), v.astype(np.float32)) for k, v in out.items()},
                 coords={"y": template.y, "x": template.x}).rio.write_crs(GRID["crs"])
dem.attrs.update({"grid": json.dumps(GRID), "params": json.dumps(PARAMS, default=list),
                  "n_tiles": int(done), "n_failed": len(failed)})
# NOTE: the arrays are in memory on the notebook server. Write them directly (zarr chunks set through
# encoding), not through .chunk(): with a Gateway client active, a Dask write would ship the data to
# the remote cluster, and the workers cannot reach a local STORE_ROOT.
dem.to_zarr(store_path(OUTPUT_NAME), mode="w", consolidated=True,
            encoding={v: {"chunks": (1024, 1024)} for v in dem.data_vars})
print(f"Wrote {OUTPUT_NAME}: lidar covers {np.nanmean(out['cov'] > 0.5):.1%} of the grid")

#### What the next cell does

It draws a visual check of three output layers:

- **Slope anomaly**, which highlights scarps and other short, steep steps.
- **Linear-feature coherence**, which is bright where the terrain has a strong single direction.
- **Detrended elevation**, which shows local steps with the broad valleys and ranges removed.

The colors are stretched from the 2nd to the 98th percentile. Areas without lidar are blank.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)
for ax, n in zip(axes, ["dem_slope_anom_max", "dem_lin_coherence", "dem_elev_detrended"]):
    v = dem[n].values
    lo, hi = np.nanpercentile(v, [2, 98])
    ax.imshow(v, vmin=lo, vmax=hi); ax.set_title(n); ax.set_axis_off()

#### What the next cell does

It disconnects from the cluster and shuts it down. The Gateway then stops all workers, so they don't keep using your compute quota.

Run this cell once the zarr file is written. If you want to process more tiles first, don't run it yet; otherwise, you'll need to start the cluster again in section 4.

In [ ]:
client.close()
cluster.close()   # NOTE: shut down the Gateway cluster so that idle workers do not keep running